# BERT for Sentiment Analysis => fine-tuning a pre-trained BERT

In [33]:
#%pip install transformers -U

In [34]:
import transformers
import tensorflow as tf

# Downloading large review movie dataset (25000 reviews)

In [35]:
!wget https://thome.isir.upmc.fr/classes/RITAL/json_pol.json

--2026-02-27 08:57:22--  https://thome.isir.upmc.fr/classes/RITAL/json_pol.json
Resolving thome.isir.upmc.fr (thome.isir.upmc.fr)... 134.157.18.247
Connecting to thome.isir.upmc.fr (thome.isir.upmc.fr)|134.157.18.247|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32663547 (31M) [application/json]
Saving to: ‘json_pol.json.2’

json_pol.json.2     100%[===================>]  31.15M  14.5MB/s    in 2.2s    

2026-02-27 08:57:26 (14.5 MB/s) - ‘json_pol.json.2’ saved [32663547/32663547]



In [36]:
import json
from collections import Counter

# Loading json
file = './json_pol.json'
with open(file,encoding="utf-8") as f:
    data = json.load(f)


# Quick Check
counter = Counter((x[1] for x in data))
print("Number of reviews : ", len(data))
print("----> # of positive : ", counter[1])
print("----> # of negative : ", counter[0])
print("")
print(data[0])


Number of reviews :  25000
----> # of positive :  12500
----> # of negative :  12500

['Although credit should have been given to Dr. Seuess for stealing the story-line of "Horton Hatches The Egg", this was a fine film. It touched both the emotions and the intellect. Due especially to the incredible performance of seven year old Justin Henry and a script that was sympathetic to each character (and each one\'s predicament), the thought provoking elements linger long after the tear jerking ones are over. Overall, superior acting from a solid cast, excellent directing, and a very powerful script. The right touches of humor throughout help keep a "heavy" subject from becoming tedious or difficult to sit through. Lastly, this film stands the test of time and seems in no way dated, decades after it was released.', 1]


# Getting the Tokenizer

In [37]:
#from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
#tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased')
model_name = "haisongzhang/roberta-tiny-cased"
#model_name = "bert-base-cased"

#model_name = "distilbert-base-uncased-finetuned-yelp-polarity"
#model_name = "textattack/bert-base-uncased-yelp-polarity"
#model_name = "rttl-ai/bert-base-uncased-yelp-reviews"


from transformers import AutoTokenizer#, BertForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Experiment the Tokenizer on the first train review

In [ ]:
maxL = 256 # Max length of the sequence

string_tokenized = tokenizer(
    data[0][0],
    return_tensors="pt",
    max_length=maxL,
    truncation=True,
    padding="max_length"
)

The output of the tokenizer string_tokenized (class BatchEncoding) returns two elements:


*   string_tokenized['input_ids']: the index of each token in the dictionary
*   string_tokenized['attention_mask']: a binary mask (0 to ignore the token, 1 to consider it). This is because we need tensor a fixed length and we have reviews with a variable number of words



In [39]:
#print(string_tokenized['input_ids'])
#print(string_tokenized['attention_mask'])

**You can use the BERT model for directly predicting polarity.** Let us apply that on the first review which has been tokenized with string_tokenized.

# Let's tokenize the whole dataset

In [ ]:
import numpy as np
from tqdm import tqdm

maxL = 256


inputs_tokens = []
attention_masks = []

for i in tqdm(range(len(data))):
    # if i % 2500 == 0:
    #     print(i)
    string_tokenized = tokenizer(
        data[i][0],
        return_tensors="pt",
        max_length=maxL,
        truncation=True,
        padding="max_length",
    )

    inputs_tokens.append(string_tokenized["input_ids"])
    attention_masks.append(string_tokenized["attention_mask"])

100%|██████████| 25000/25000 [01:10<00:00, 354.91it/s]


# Let's create a 'TensorDataSet' FOR THE SAMPLES where each element is a triplet composed of token word index, token mask, and label

In [43]:
#print(data[1300][1])
#print(y[0])

In [44]:
import torch
# Converting input tokens to torch tensors
inputs_tokens = torch.cat(inputs_tokens, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)


In [45]:
# Converting labels to torch tensor
#y = torch.zeros((len(data),2), dtype=torch.float)
y = torch.zeros((len(data),), dtype=torch.float)

for i in range(len(data)):
    y[i] = data[i][1]
    #y[i][data[i][1]] = 1
#y = torch.from_numpy(y)

In [46]:
print(inputs_tokens.shape)
print(attention_masks.shape)
print(y.shape)

torch.Size([25000, 512])
torch.Size([25000, 512])
torch.Size([25000])


In [47]:
from sklearn.model_selection import train_test_split

np.random.seed(0)
rs=10

inputs_tokens_train, inputs_tokens_test, attention_masks_train, attention_masks_test, y_train, y_test =train_test_split(inputs_tokens, attention_masks, y, test_size=0.5, random_state=rs)

print(inputs_tokens_train.shape)
print(inputs_tokens_test.shape)

print(attention_masks_train.shape)
print(attention_masks_test.shape)

print(y_train.shape)
print(y_test.shape)

torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500])
torch.Size([12500])


In [48]:
#print(y_train[10000])

In [49]:
from torch.utils.data import TensorDataset, random_split, DataLoader, RandomSampler, SequentialSampler

dataset_train = TensorDataset(inputs_tokens_train,  attention_masks_train, y_train)
dataset_test = TensorDataset(inputs_tokens_test,  attention_masks_test, y_test)

# Lets download a BERT model for word embedding

In [50]:
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: haisongzhang/roberta-tiny-cased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [51]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-12, e

In [52]:
#train_dataloader = DataLoader(dataset_train, batch_size=2,shuffle=True)

#it, tm, labelsb = next(iter(train_dataloader))
#
#pred = model(input_ids=it, attention_mask=tm)
#print(pred[0])

In [53]:
#print(pred.logits)

# FINE-TUNING THE MODEL

In [54]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [55]:
def accuracy(model, dataloader):
  model.eval()
  nbgood =0
  for idx,batch in enumerate(dataloader):
    b_input_ids = batch[0].cuda()
    b_input_mask = batch[1].cuda()
    b_labels = batch[2].cuda()

    with torch.no_grad():
      pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
      yhat = pred.logits.argmax(axis=1)
      ytrue = b_labels
      nbgood += (yhat==ytrue).sum()

    if(idx>0 and idx%250==0):
      print("idx=",idx," nbgood=",nbgood.item())

  acc = nbgood / 125.0
  return acc.item()


In [56]:
def accuracy_batch(model, b_input_ids, b_input_mask, b_labels):
  model.eval()

  with torch.no_grad():
    pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
    yhat = pred.logits.argmax(axis=1)
    ytrue = b_labels

    print(yhat.detach().cpu().numpy())
    print(ytrue.detach().cpu().numpy())

    acc = (yhat==ytrue).sum()*4.0

  return acc.item()


In [59]:
# create DataLoaders with samplers

import torch.nn as nn
import torch.optim as optim
tb = int(25)
train_dataloader = DataLoader(dataset_train, batch_size=tb,shuffle=True)
test_dataloader = DataLoader(dataset_test, batch_size=tb,shuffle=False)
nbTrain = len(data)

temb = 768
temb = 512
#features = np.zeros((nbTrain, temb))
#nbtach = int(nbTrain/tb)
#print(f"nb batches={nbtach}")

#nbgood =torch.tensor(0.0).cuda()
nbepochs =5
loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Comuting CLS features
model.train()
model.cuda()

for e in range(nbepochs):
    for idx,batch in tqdm(enumerate(train_dataloader)):
        b_input_ids = batch[0].cuda()
        b_input_mask = batch[1].cuda()
        b_labels = batch[2].cuda().long()
        
        optimizer.zero_grad()
        
        pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
        #print(b_labels)
        ce = loss(pred.logits,b_labels)
        ce.backward()
        optimizer.step()
        
        if(idx>0 and idx%250==0):
            print("** batch: ",idx," acc train=",accuracy(model,train_dataloader)," acc test=",accuracy(model,test_dataloader) )
    
    print("**** epoch",(e+1)," acc train=",accuracy(model,train_dataloader)," acc test=",accuracy(model,test_dataloader) )



250it [01:49,  2.22it/s]

idx= 250  nbgood= 5712
idx= 250  nbgood= 5453


252it [04:05, 28.62s/it]

** batch:  250  acc train= 90.46400451660156  acc test= 87.0560073852539


500it [05:55,  1.41it/s]


...

idx= 250  nbgood= 6225
idx= 250  nbgood= 5529


251it [04:08, 41.46s/it]

** batch:  250  acc train= 99.22400665283203  acc test= 88.4000015258789


500it [06:00,  1.39it/s]


idx= 250  nbgood= 6225
idx= 250  nbgood= 5551
**** epoch 5  acc train= 99.22400665283203  acc test= 88.68000793457031
